In [1]:
!pip install torch transformers datasets numpy pandas scikit-learn matplotlib tqdm



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from whitebox_uncertainty import WhiteBoxUncertainty

In [7]:
from transformers import AutoTokenizer
import torch

# Hazır tokenizer
tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-turkish-uncased")

# Metin örneği
text = "merhaba"

# Tokenizer ile encode
token_ids = tokenizer.encode(text, add_special_tokens=True)  # [CLS] ve [SEP] tokenleri de eklenebilir

# Örnek logits
logits_t1 = torch.tensor([0.1, 0.5, 0.4])
logits_t2 = torch.tensor([0.3, 0.4, 0.3])
logits_t3 = torch.tensor([0.2, 1, 3])

scores = [logits_t1, logits_t2, logits_t3]

# WhiteBoxUncertainty örneği
uncertainty = WhiteBoxUncertainty(
    scores=scores,
    text_responses=text,
    token_ids=token_ids
)

# Self-consistency hesabı
consistency = uncertainty.self_consistency()
print("Consistency:", consistency)


Consistency: 0.2857142857142857


In [9]:
from graybox_uncertainty import GrayBoxUncertainty
import torch

responses = [
    "Paris is the capital of France",
    " France",
    "Paris ",
    "Paris is the capital of France",
    "Paris is the capital of France"
]

# log_probs listesi (her eleman tensor olmalı)
log_probs = [
    torch.tensor([0.1, 0.3, 0.5]),
    torch.tensor([0.2, 0.2, 0.6]),
    torch.tensor([0.3, 0.4, 0.3]),
    torch.tensor([0.1, 0.3, 0.6]),
    torch.tensor([0.2, 0.5, 0.3])
]

# GrayBoxUncertainty örneğini oluştur
gbu = GrayBoxUncertainty(responses=responses, log_probs=log_probs)

print("Consistency:", gbu.self_consistency())
print("Entropy:", gbu.response_entropy())
print("Mean log prob:", gbu.mean_log_probability())
print("Confidence:", gbu.confidence())


Consistency: 0.6
Entropy: 0.9502705392302345
Mean log prob: 0.3266666829586029
Confidence: 0.8318035857860161


In [10]:
from black_uncertainty import BlackBoxUncertainty
responses = [
    "Paris is the capital of France",
    "Paris is the capital of France",
    "Paris",
    "Paris is the capital of France",
    "France's capital is Paris"
]

bb = BlackBoxUncertainty(responses)

print("Self-consistency:", bb.self_consistency())
print("Response entropy:", bb.response_entropy())
print("Unique ratio:", bb.unique_ratio())
print("Confidence:", bb.confidence())


Self-consistency: 0.6
Response entropy: 0.9502705392302345
Unique ratio: 0.6
Confidence: 0.6


In [ ]:
!pip install sentence-transformers



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
from semantic_uncertainty import EnsembleSemanticUncertainty
responses = [
    "Paris is the capital of France",
    "France's capital city is Paris",
    "Paris is a major city in Europe"
]

semantic = EnsembleSemanticUncertainty(responses, language="en")

print("Semantic Consistency:", semantic.semantic_consistency())
print("Semantic Uncertainty:", semantic.uncertainty())
print("Per-model:", semantic.per_model_scores())


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/python/3.12.1/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 480.56it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00,

Semantic Consistency: 0.8632496913274128
Semantic Uncertainty: 0.1367503086725872
Per-model: {'sentence-transformers/all-MiniLM-L6-v2': 0.8452528913815817, 'sentence-transformers/all-mpnet-base-v2': 0.8045051495234171, 'sentence-transformers/paraphrase-MiniLM-L12-v2': 0.9110526442527771, 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1': 0.8457762400309244, 'sentence-transformers/paraphrase-mpnet-base-v2': 0.9096615314483643}


In [2]:
responses_tr = [
    "Paris Fransa'nın başkentidir",
    "Fransa'nın başkenti Paris'tir",
    "Paris Avrupa'da bir şehirdir"
]

semantic_tr = EnsembleSemanticUncertainty(responses_tr, language="tr")

print("TR Semantic Consistency:", semantic_tr.semantic_consistency())
print("TR Semantic Uncertainty:", semantic_tr.uncertainty())


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 427.13it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 213.06it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 505.59it/s, Material

TR Semantic Consistency: 0.8573737382888794
TR Semantic Uncertainty: 0.14262626171112058
